# Retries
Retry temporary failures with a limit and delay.


In [ ]:
# Retry a temporary connection failure, but stop after a fixed limit.
import asyncio

attempts = 0
async def unstable() -> str:
    global attempts
    attempts += 1
    if attempts < 3:
        raise ConnectionError
    return "ok"

for attempt in range(3):
    try:
        print(await unstable())
        break
    except ConnectionError:
        await asyncio.sleep(0.01)


## Polished version
Retry only known transient errors and use exponential backoff.


In [ ]:
# A reusable policy decides which errors are safe to retry.
from collections.abc import Awaitable, Callable
from dataclasses import dataclass
from typing import TypeVar

T = TypeVar("T")

class ProviderHttpError(Exception):
    def __init__(self, status_code: int) -> None:
        self.status_code = status_code

# Client errors are usually permanent; timeouts, connections, and 5xx are transient.
def is_retryable(error: Exception) -> bool:
    return isinstance(error, (TimeoutError, ConnectionError)) or (
        isinstance(error, ProviderHttpError) and error.status_code >= 500
    )

@dataclass(frozen=True)
class RetryPolicy:
    attempts: int = 3
    base_delay: float = 0.01

    async def run(self, operation: Callable[[], Awaitable[T]]) -> T:
        for attempt in range(self.attempts):
            try:
                return await operation()
            except Exception as error:
                if attempt == self.attempts - 1 or not is_retryable(error):
                    raise
                # Exponential backoff reduces pressure on a struggling dependency.
                await asyncio.sleep(self.base_delay * 2**attempt)
        raise RuntimeError("unreachable")

attempts = 0
print(await RetryPolicy().run(unstable))
